In [1]:
#resnet练习
import matplotlib as mpl
import matplotlib.pyplot as plt
%matplotlib inline
import numpy as np
import sklearn
import pandas as pd
import os
import sys
import time
from tqdm.auto import tqdm
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, transforms
import matplotlib.pyplot as plt

plt.tight_layout()
plt.show()

device = torch.device("cuda:0") if torch.cuda.is_available() else torch.device("cpu")
print(device)



data_dir = './archive'

transform = transforms.Compose([transforms.ToTensor(),
                                transforms.Resize((128, 128)),
                                transforms.Normalize(mean=[0.4363, 0.4328, 0.3291], std=[0.2427, 0.2382, 0.2413])
                                ])

train_dataset = datasets.ImageFolder(root = os.path.join(data_dir, 'training'), transform=transform)
test_dataset = datasets.ImageFolder(root = os.path.join(data_dir, 'validation'), transform=transform)

class_names = train_dataset.classes
print(class_names)

<Figure size 640x480 with 0 Axes>

cuda:0
['n0', 'n1', 'n2', 'n3', 'n4', 'n5', 'n6', 'n7', 'n8', 'n9']


In [2]:
from torch.utils.data import DataLoader

batch_size = 32

train_loader = DataLoader(train_dataset,batch_size=batch_size,shuffle=True,num_workers=4)
test_loader = DataLoader(test_dataset,batch_size=batch_size,shuffle=False,num_workers=4)

In [3]:
import torch.nn as nn
from torchvision.models import resnet50

class ResNet50(nn.Module):
    def __init__(self,num_classes=10,frozen=True):
        super().__init__()
        
        self.model = resnet50(weights='IMAGENET1K_V2')
        
        if frozen:
            for param in self.model.parameters():
                param.requires_grad = False
        
        for param in self.model.layer4.parameters():
            param.requires_grad = True
        
        in_features = self.model.fc.in_features
        self.model.fc = nn.Linear(in_features,num_classes)
        
    def forward(self,x):
        return self.model(x)

model = ResNet50()


Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to C:\Users\octopus/.cache\torch\hub\checkpoints\resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:49<00:00, 2.05MB/s]


In [4]:
import torch

dummy_input = torch.randn(32, 3, 128, 128)
output = model(dummy_input)
print(output.shape)

total_params = 0
for name, param in model.named_parameters():
    if param.requires_grad:
        num_params = param.numel()
        total_params += num_params
        print(name,':',num_params)
print(f"Total Parameters: {total_params}")


torch.Size([32, 10])
model.layer4.0.conv1.weight : 524288
model.layer4.0.bn1.weight : 512
model.layer4.0.bn1.bias : 512
model.layer4.0.conv2.weight : 2359296
model.layer4.0.bn2.weight : 512
model.layer4.0.bn2.bias : 512
model.layer4.0.conv3.weight : 1048576
model.layer4.0.bn3.weight : 2048
model.layer4.0.bn3.bias : 2048
model.layer4.0.downsample.0.weight : 2097152
model.layer4.0.downsample.1.weight : 2048
model.layer4.0.downsample.1.bias : 2048
model.layer4.1.conv1.weight : 1048576
model.layer4.1.bn1.weight : 512
model.layer4.1.bn1.bias : 512
model.layer4.1.conv2.weight : 2359296
model.layer4.1.bn2.weight : 512
model.layer4.1.bn2.bias : 512
model.layer4.1.conv3.weight : 1048576
model.layer4.1.bn3.weight : 2048
model.layer4.1.bn3.bias : 2048
model.layer4.2.conv1.weight : 1048576
model.layer4.2.bn1.weight : 512
model.layer4.2.bn1.bias : 512
model.layer4.2.conv2.weight : 2359296
model.layer4.2.bn2.weight : 512
model.layer4.2.bn2.bias : 512
model.layer4.2.conv3.weight : 1048576
model.layer

In [5]:
import torch.nn as nn
import torch.optim as optim
import my_trainer as mt

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

trainer = mt.Trainer(
    model=model,
    train_loader=train_loader,
    val_loader=test_loader,
    criterion=criterion,
    optimizer=optimizer,
    device=device,
    eval_step=100
)

num_epochs = 20
trainer.train(num_epochs)

Epoch [1/20]  Train Loss: 0.6200  Train Acc: 0.8350
Epoch [2/20]  Train Loss: 0.1420  Train Acc: 0.9699
[Step 100] Val Loss: 0.1285 Val Acc: 0.9449
Epoch [3/20]  Train Loss: 0.0431  Train Acc: 0.9909
Epoch [4/20]  Train Loss: 0.0207  Train Acc: 0.9927
Epoch [5/20]  Train Loss: 0.0372  Train Acc: 0.9872
[Step 200] Val Loss: 0.1599 Val Acc: 0.9559
Epoch [6/20]  Train Loss: 0.0758  Train Acc: 0.9836
Epoch [7/20]  Train Loss: 0.0390  Train Acc: 0.9900
Epoch [8/20]  Train Loss: 0.0336  Train Acc: 0.9945
[Step 300] Val Loss: 0.0509 Val Acc: 0.9816
Epoch [9/20]  Train Loss: 0.0506  Train Acc: 0.9863
Epoch [10/20]  Train Loss: 0.0539  Train Acc: 0.9863
Epoch [11/20]  Train Loss: 0.0464  Train Acc: 0.9909
[Step 400] Val Loss: 0.0759 Val Acc: 0.9853
Epoch [12/20]  Train Loss: 0.0569  Train Acc: 0.9954
Epoch [13/20]  Train Loss: 0.0062  Train Acc: 0.9991
Epoch [14/20]  Train Loss: 0.0030  Train Acc: 1.0000
[Step 500] Val Loss: 0.0806 Val Acc: 0.9816
Epoch [15/20]  Train Loss: 0.0182  Train Acc: 0